In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'Ready in: {os.getcwd()}')


In [ ]:
import numpy as np
import pandas as pd
import json, time, joblib, traceback
from pathlib import Path
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

SEED = 42
np.random.seed(SEED)

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
ARCHITECTURES = ['rf', 'xgb', 'dnn']
VARIANTS = ['5class_cw', '5class_smote']
MODELS_PER_DATASET = [f'{a}_{v}' for v in VARIANTS for a in ARCHITECTURES]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
PROTOCOLS = ['v2', 'strict']

# Stability (verbatim from 05c)
PERTURBATIONS = ['gaussian', 'fgsm', 'pgd']
ATTACK_SOURCE_MODEL = 'dnn_5class_cw'
EPSILON, PGD_ALPHA, PGD_STEPS = 0.05, 0.005, 10
TOP_K = 10

# Conformal (verbatim from 07c / 07e)
ALPHAS = [0.05, 0.10, 0.20]
ALPHA_PRIMARY = 0.05
MIN_CALIB_MONDRIAN = 30
EPS = 1e-6

# Strict protocol (verbatim from 07e)
HOLDOUT_FRAC = 0.20
RARE_CLASS_THRESHOLD = 25
PLATT_THRESHOLD = 30

# Health flag (verbatim from 07d)
T_GREEN_LO, T_GREEN_HI = 0.05, 0.95
T_RED_HI, T_RED_LO = 0.999, 0.001
CLIFF_GREEN_HI, CLIFF_RED_LO, CLIFF_THRESH = 0.05, 0.20, 0.95
N_GREEN_LO, N_RED_HI = 100, 30

# Bootstrap
N_BOOTSTRAP = 10000
BOOTSTRAP_SEED = 42
MIN_MINORITY_FOR_CI = 10   # cells with fewer misclassified (or correct) samples get point estimates only

SCORES = ['c1', 'c2', 'c3', 'scts', 'scts_c1c2', 'scts_c1c3']
FLAG_ORDER = ['green', 'amber', 'red']

TABLES = Path(REPO) / 'results' / 'tables'
DOCS = Path(REPO) / 'docs'
TABLES.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

PREFIX = 'trust_ablation'
print(f'{len(DATASETS)} datasets x {len(MODELS_PER_DATASET)} models x {len(PROTOCOLS)} protocols = '
      f'{len(DATASETS) * len(MODELS_PER_DATASET) * len(PROTOCOLS)} cells; B={N_BOOTSTRAP}')


In [ ]:
def find_proba_file(dataset, model_name, split):
    fname = f'{model_name}_{split}_proba.npy'
    for subdir in ['probabilities', 'predictions']:
        p = Path(REPO) / 'models' / dataset / subdir / fname
        if p.exists():
            return p
    raise FileNotFoundError(f'No {fname} for {dataset}/{model_name}')

def find_tree_model_path(dataset, model_name):
    base = Path(REPO) / 'models' / dataset
    for ext in ['.pkl', '.joblib']:
        p = base / f'{model_name}{ext}'
        if p.exists():
            return p
    raise FileNotFoundError(f'No tree model file for {model_name} in {base}')

def find_dnn_path(dataset, model_name):
    p = Path(REPO) / 'models' / dataset / f'{model_name}.pt'
    if p.exists():
        return p
    raise FileNotFoundError(f'No DNN file at {p}')

def shap_mod():
    try:
        import shap
    except ImportError:
        os.system('pip install -q shap')
        import shap
    return shap

_torch = {}
def torch_env():
    # Imported on demand: only needed when a perturbation or perturbed-SHAP cache is missing.
    if not _torch:
        import torch, torch.nn as nn, torch.nn.functional as F
        _torch['torch'], _torch['nn'], _torch['F'] = torch, nn, F
        _torch['DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
        torch.manual_seed(SEED)

        class DNN(nn.Module):
            def __init__(self, in_dim, n_classes, hidden=(256, 128, 64, 32), dropout=0.3):
                super().__init__()
                layers, prev = [], in_dim
                for h in hidden:
                    layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
                    prev = h
                layers.append(nn.Linear(prev, n_classes))
                self.net = nn.Sequential(*layers)
            def forward(self, x):
                return self.net(x)

        class DNNWithSoftmax(nn.Module):
            def __init__(self, dnn):
                super().__init__()
                self.dnn = dnn
            def forward(self, x):
                return torch.softmax(self.dnn(x), dim=-1)

        _torch['DNN'], _torch['DNNWithSoftmax'] = DNN, DNNWithSoftmax
    return _torch

def load_pytorch_dnn(path, return_raw=False):
    T = torch_env(); torch = T['torch']
    ckpt = torch.load(path, map_location=T['DEVICE'], weights_only=False)
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        in_dim, n_classes = ckpt['in_dim'], ckpt['n_classes']
        hidden, dropout, state_dict = tuple(ckpt['hidden']), ckpt['dropout'], ckpt['state_dict']
    else:
        state_dict = ckpt
        in_dim = state_dict['net.0.weight'].shape[1]
        n_classes = state_dict['net.16.weight'].shape[0]
        hidden, dropout = (256, 128, 64, 32), 0.3
    model = T['DNN'](in_dim=in_dim, n_classes=n_classes, hidden=hidden, dropout=dropout)
    model.load_state_dict(state_dict)
    model = model.to(T['DEVICE']).eval()
    if return_raw:
        return model
    return T['DNNWithSoftmax'](model).to(T['DEVICE']).eval()

def gaussian_perturbation(X, epsilon, seed):
    rng = np.random.RandomState(seed)
    return (X + rng.normal(0, epsilon, X.shape)).astype(np.float32)

def fgsm_attack(model_raw, X, y, epsilon):
    T = torch_env(); torch, F = T['torch'], T['F']
    model_raw.eval()
    X_t = torch.tensor(X, dtype=torch.float32, requires_grad=True, device=T['DEVICE'])
    y_t = torch.tensor(y, dtype=torch.long, device=T['DEVICE'])
    loss = F.cross_entropy(model_raw(X_t), y_t)
    grad = torch.autograd.grad(loss, X_t)[0]
    return (X_t + epsilon * grad.sign()).detach().cpu().numpy().astype(np.float32)

def pgd_attack(model_raw, X, y, epsilon, alpha, steps, seed=None):
    # L-infinity projection (clamp to X +/- epsilon), identical to 05c.
    T = torch_env(); torch, F = T['torch'], T['F']
    model_raw.eval()
    X_t = torch.tensor(X, dtype=torch.float32, device=T['DEVICE'])
    y_t = torch.tensor(y, dtype=torch.long, device=T['DEVICE'])
    if seed is not None:
        torch.manual_seed(seed)
    X_adv = X_t + torch.empty_like(X_t).uniform_(-epsilon, epsilon)
    for _ in range(steps):
        X_adv = X_adv.detach().requires_grad_(True)
        loss = F.cross_entropy(model_raw(X_adv), y_t)
        grad = torch.autograd.grad(loss, X_adv)[0]
        X_adv = X_adv + alpha * grad.sign()
        X_adv = torch.max(torch.min(X_adv, X_t + epsilon), X_t - epsilon)
    return X_adv.detach().cpu().numpy().astype(np.float32)

def ensure_perturbations(ds):
    pert_dir = Path(REPO) / 'shap_values' / ds / 'perturbations'
    pert_dir.mkdir(parents=True, exist_ok=True)
    paths = {p: pert_dir / f'{p}_X.npy' for p in PERTURBATIONS}
    if all(p.exists() for p in paths.values()):
        return paths
    eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    X_test = np.load(f'{REPO}/data/processed/{ds}/X_test.npy').astype(np.float32)
    y_test = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')
    X_c, y_c = X_test[eval_idx], y_test[eval_idx]
    if not paths['gaussian'].exists():
        np.save(paths['gaussian'], gaussian_perturbation(X_c, EPSILON, seed=SEED))
    if not paths['fgsm'].exists() or not paths['pgd'].exists():
        dnn_raw = load_pytorch_dnn(find_dnn_path(ds, ATTACK_SOURCE_MODEL), return_raw=True)
        if not paths['fgsm'].exists():
            np.save(paths['fgsm'], fgsm_attack(dnn_raw, X_c, y_c, EPSILON))
        if not paths['pgd'].exists():
            np.save(paths['pgd'], pgd_attack(dnn_raw, X_c, y_c, EPSILON, PGD_ALPHA, PGD_STEPS, seed=SEED))
    return paths

def _to_nfc(shap_values, n_samples, n_features, n_classes=5):
    # Normalise SHAP output to (n_samples, n_features, n_classes), identical to 04c/05c handling.
    if isinstance(shap_values, list):
        shap_values = np.stack(shap_values, axis=-1)
    if shap_values.shape == (n_classes, n_samples, n_features):
        shap_values = np.transpose(shap_values, (1, 2, 0))
    elif shap_values.shape == (n_samples, n_classes, n_features):
        shap_values = np.transpose(shap_values, (0, 2, 1))
    assert shap_values.shape == (n_samples, n_features, n_classes), shap_values.shape
    return shap_values

def ensure_perturbed_shap(ds, model_name, perturbation, X_pert, bg_X):
    out_path = Path(REPO) / 'shap_values' / ds / 'perturbed_shap' / f'{model_name}_{perturbation}_shap.npy'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        return out_path, True
    shap = shap_mod()
    n, f = X_pert.shape
    if model_name.startswith(('rf', 'xgb')):
        model = joblib.load(find_tree_model_path(ds, model_name))
        sv = shap.TreeExplainer(model).shap_values(X_pert, check_additivity=False)
    else:
        T = torch_env(); torch = T['torch']
        wrapped = load_pytorch_dnn(find_dnn_path(ds, model_name), return_raw=False)
        bg_t = torch.from_numpy(bg_X).to(T['DEVICE'])
        ev_t = torch.from_numpy(X_pert).to(T['DEVICE'])
        sv = shap.GradientExplainer(wrapped, bg_t).shap_values(ev_t)
        del wrapped, bg_t, ev_t
        if T['DEVICE'] == 'cuda':
            torch.cuda.empty_cache()
    np.save(out_path, _to_nfc(sv, n, f))
    return out_path, False

def topk_sets(shap_arr, k=TOP_K):
    # Per-sample importance = sum over classes of |SHAP|, identical to 05c.
    imp = np.abs(shap_arr).sum(axis=-1) if shap_arr.ndim == 3 else np.abs(shap_arr)
    return np.argsort(-imp, axis=1)[:, :k]

def jaccard_and_retention(shap_orig, shap_pert, k=TOP_K):
    # jaccard   = |A & B| / |A | B|      (the quantity 05c computed and 07c used as c2)
    # retention = |A & B| / k            (the quantity the manuscript text describes)
    A, B = topk_sets(shap_orig, k), topk_sets(shap_pert, k)
    inter = np.array([len(set(a) & set(b)) for a, b in zip(A, B)], dtype=np.float64)
    return inter / (2 * k - inter), inter / k

print('Model, perturbation and SHAP helpers ready')


In [ ]:
def split_conformal_threshold(probs, y_true, alpha):
    n = len(y_true)
    scores = 1.0 - probs[np.arange(n), y_true]
    q = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(scores, q))

def mondrian_conformal_thresholds(probs, y_true, alpha, n_classes=5, min_calib=MIN_CALIB_MONDRIAN):
    # Stratified by PREDICTED class; falls back to the marginal threshold when n_c < min_calib.
    y_pred = probs.argmax(axis=1)
    marginal = split_conformal_threshold(probs, y_true, alpha)
    thresholds, fallback, n_per_class = {}, [], {}
    for c in range(n_classes):
        mask = y_pred == c
        n_c = int(mask.sum())
        n_per_class[c] = n_c
        if n_c < min_calib:
            thresholds[c] = marginal
            fallback.append(c)
        else:
            s = 1.0 - probs[mask, :][np.arange(n_c), y_true[mask]]
            q = min(np.ceil((n_c + 1) * (1 - alpha)) / n_c, 1.0)
            thresholds[c] = float(np.quantile(s, q))
    return thresholds, fallback, n_per_class

def empirical_coverage_mondrian(probs, y_true, thresholds):
    y_pred = probs.argmax(axis=1)
    s = 1.0 - probs[np.arange(len(y_true)), y_true]
    t = np.array([thresholds[int(p)] for p in y_pred])
    return float((s <= t).mean())

def component_3_safe(probs, y_pred, thresholds, eps=1e-9):
    # c3 = clip(1 - s / tau_pred, 0, 1) with s = 1 - p_hat(y_pred), identical to 07c;
    # zero-threshold handling identical to 07f (c3 = 1 iff s <= eps).
    n = len(y_pred)
    t = np.array([thresholds[int(p)] for p in y_pred], dtype=np.float64)
    s = 1.0 - probs[np.arange(n), y_pred]
    c3 = np.zeros(n, dtype=np.float64)
    nz = t > eps
    c3[nz] = np.clip(1.0 - s[nz] / t[nz], 0.0, 1.0)
    c3[~nz] = (s[~nz] <= eps).astype(np.float64)
    return c3

def geo_mean(*comps):
    g = np.ones_like(comps[0], dtype=np.float64)
    for c in comps:
        g = g * np.clip(c, EPS, 1.0)
    return g ** (1.0 / len(comps))

def cliff_fractions(probs, y_true):
    # Signal 2 surface: fraction of samples predicted as c whose true-class probability is <= 1 - CLIFF_THRESH.
    scores = 1.0 - probs[np.arange(len(y_true)), y_true]
    y_pred = probs.argmax(axis=1)
    out = {}
    for c in range(5):
        m = y_pred == c
        out[c] = float('nan') if m.sum() == 0 else float((scores[m] >= CLIFF_THRESH).mean())
    return out

def fit_calibrator(p_calib, y_indicator, n_class):
    if n_class >= PLATT_THRESHOLD:
        cal = IsotonicRegression(out_of_bounds='clip')
        cal.fit(p_calib, y_indicator)
        return cal, 'isotonic'
    cal = LogisticRegression(C=1e10, solver='lbfgs')
    cal.fit(p_calib.reshape(-1, 1), y_indicator)
    return cal, 'platt'

def apply_calibrator(cal, strategy, p):
    if strategy == 'isotonic':
        return cal.predict(p)
    return cal.predict_proba(p.reshape(-1, 1))[:, 1]

def strict_split(y_calib):
    # Class-aware 80/20 split, identical to 07e: rare classes (n < 25) stay entirely in the strict slice;
    # iteration order follows Counter insertion order so the RNG stream matches 07e exactly.
    n = len(y_calib)
    rng = np.random.RandomState(SEED)
    counts = Counter(y_calib.tolist())
    rare = [c for c, k in counts.items() if k < RARE_CLASS_THRESHOLD]
    common = [c for c, k in counts.items() if k >= RARE_CLASS_THRESHOLD]
    mask = np.zeros(n, dtype=bool)
    for c in rare:
        mask[y_calib == c] = True
    for c in common:
        idx = np.where(y_calib == c)[0]
        rng.shuffle(idx)
        mask[idx[:int(round(len(idx) * (1 - HOLDOUT_FRAC)))]] = True
    return np.where(mask)[0], np.where(~mask)[0], rare

def flag_threshold(t):
    if t >= T_RED_HI or t < T_RED_LO: return 'red'
    if t >= T_GREEN_HI or t <= T_GREEN_LO: return 'amber'
    return 'green'

def flag_cliff(f):
    if np.isnan(f): return 'red'
    if f >= CLIFF_RED_LO: return 'red'
    if f >= CLIFF_GREEN_HI: return 'amber'
    return 'green'

def flag_support(n):
    if n < N_RED_HI: return 'red'
    if n < N_GREEN_LO: return 'amber'
    return 'green'

def combine_flags(*flags):
    if 'red' in flags: return 'red'
    if 'amber' in flags: return 'amber'
    return 'green'

def wilson(k, n, z=1.959964):
    if n == 0:
        return (float('nan'), float('nan'))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (float(c - h), float(c + h))

print('Conformal, calibrator and flag helpers ready')


In [ ]:
def boot_index(n, B=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    return np.random.RandomState(seed).randint(0, n, size=(B, n))

def boot_weights(idx, n):
    # Multiplicity matrix (B, n): how many times each sample appears in each resample.
    B = idx.shape[0]
    flat = (np.arange(B)[:, None] * n + idx).ravel()
    return np.bincount(flat, minlength=B * n).reshape(B, n).astype(np.float64)

def pearson_w(W, x, y):
    # Pearson of the resampled vectors, computed from multiplicities without materialising the resamples.
    sw = W.sum(axis=1)
    sx, sy = W @ x, W @ y
    sxx, syy, sxy = W @ (x * x), W @ (y * y), W @ (x * y)
    with np.errstate(invalid='ignore', divide='ignore'):
        return (sxy - sx * sy / sw) / np.sqrt((sxx - sx * sx / sw) * (syy - sy * sy / sw))

def auroc_w(W, x, y):
    # Mann-Whitney AUROC of the resampled vectors with tie handling (0.5 credit), from multiplicities.
    order = np.argsort(x, kind='stable')
    xs, ys = x[order], y[order]
    starts = np.flatnonzero(np.r_[True, xs[1:] != xs[:-1]])
    Ws = W[:, order]
    Wp = np.add.reduceat(Ws * ys, starts, axis=1)
    Wn = np.add.reduceat(Ws * (1.0 - ys), starts, axis=1)
    below = np.cumsum(Wn, axis=1) - Wn
    with np.errstate(invalid='ignore', divide='ignore'):
        return (Wp * (below + 0.5 * Wn)).sum(axis=1) / (Wp.sum(axis=1) * Wn.sum(axis=1))

def pct_ci(v):
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return (float('nan'), float('nan'), 0)
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)), int(len(v)))

def cluster_boot_rate(mask_num, mask_den, sample_pos, B=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    # Resample canonical sample POSITIONS with replacement, carrying every model's row for a resampled position.
    # This is the correct unit when the same 1000 canonical samples are scored by six models.
    positions = np.unique(sample_pos)
    n_pos = len(positions)
    num_by_pos = np.bincount(sample_pos, weights=mask_num.astype(float), minlength=sample_pos.max() + 1)[positions]
    den_by_pos = np.bincount(sample_pos, weights=mask_den.astype(float), minlength=sample_pos.max() + 1)[positions]
    idx = np.random.RandomState(seed).randint(0, n_pos, size=(B, n_pos))
    num = num_by_pos[idx].sum(axis=1)
    den = den_by_pos[idx].sum(axis=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        rates = num / den
    return pct_ci(rates)

print('Bootstrap helpers ready')


In [ ]:
t0 = time.time()
stab_rows = []
stab_log = []

for ds in DATASETS:
    eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    bg_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_bg_idx.npy')
    y_test = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')
    y_c = y_test[eval_idx]
    X_calib = np.load(f'{REPO}/data/processed/{ds}/X_calib.npy').astype(np.float32)
    bg_X = X_calib[bg_idx]
    pert_paths = ensure_perturbations(ds)
    pert_arrays = {p: np.load(pert_paths[p]) for p in PERTURBATIONS}

    for model_name in MODELS_PER_DATASET:
        shap_orig = np.load(Path(REPO) / 'shap_values' / ds / f'{model_name}_shap_shared.npy')
        for pert in PERTURBATIONS:
            t1 = time.time()
            p_path, cached = ensure_perturbed_shap(ds, model_name, pert, pert_arrays[pert], bg_X)
            shap_pert = np.load(p_path)
            jac, ret = jaccard_and_retention(shap_orig, shap_pert, TOP_K)
            for i in range(len(jac)):
                stab_rows.append({'dataset': ds, 'model': model_name, 'perturbation': pert,
                                  'sample_position': i, 'true_class': int(y_c[i]),
                                  'jaccard_top10': float(jac[i]), 'retention_top10': float(ret[i])})
            stab_log.append({'dataset': ds, 'model': model_name, 'perturbation': pert,
                             'cached': cached, 'seconds': round(time.time() - t1, 1),
                             'mean_jaccard': float(jac.mean()), 'mean_retention': float(ret.mean())})
            print(f'{ds:15s} {model_name:18s} {pert:9s} {"cached" if cached else "computed":8s} '
                  f'jaccard={jac.mean():.3f} retention={ret.mean():.3f}')

df_stab = pd.DataFrame(stab_rows)
df_stab.to_csv(TABLES / f'{PREFIX}_stability_per_sample.csv', index=False)
pd.DataFrame(stab_log).to_csv(TABLES / f'{PREFIX}_stability_log.csv', index=False)
print(f'\n{len(df_stab)} per-sample rows (expect {len(DATASETS) * 6 * 3 * 1000}); {(time.time() - t0) / 60:.1f} min')

# Consistency against the committed artefacts: the committed per-sample file (NSL only) and the
# committed per-cell means (all three datasets, from bootstrap_cis_stability.csv).
old_ps = TABLES / 'stability_v2_per_sample_jaccard.csv'
if old_ps.exists():
    o = pd.read_csv(old_ps)[['dataset', 'model', 'perturbation', 'sample_position', 'jaccard_top10']]
    m = df_stab.merge(o, on=['dataset', 'model', 'perturbation', 'sample_position'], suffixes=('', '_old'))
    print(f'vs stability_v2_per_sample_jaccard.csv: {len(m)} matched rows, '
          f'max |diff| = {np.abs(m.jaccard_top10 - m.jaccard_top10_old).max():.2e}')
old_ci = TABLES / 'bootstrap_cis_stability.csv'
if old_ci.exists():
    o = pd.read_csv(old_ci)[['dataset', 'model', 'mean_jaccard']]
    new_means = df_stab.groupby(['dataset', 'model']).jaccard_top10.mean().reset_index()
    m = new_means.merge(o, on=['dataset', 'model'])
    m['diff'] = (m.jaccard_top10 - m.mean_jaccard).abs()
    print(f'vs bootstrap_cis_stability.csv cell means: {len(m)} cells, max |diff| = {m["diff"].max():.2e}')
    print(m.sort_values('diff', ascending=False).head(3).to_string(index=False))

worst = df_stab.groupby(['dataset', 'model', 'sample_position']).agg(
    c2=('jaccard_top10', 'min'), c2_retention=('retention_top10', 'min')).reset_index()
c2_lookup = {(ds, m): g.sort_values('sample_position')[['c2', 'c2_retention']].values
             for (ds, m), g in worst.groupby(['dataset', 'model'])}
print(f'c2 lookup: {len(c2_lookup)} cells')


In [ ]:
# v2 protocol (07c/07d): calibrators from 03e applied to the full test set; Mondrian thresholds,
# cliff fractions and support counts all measured on test-outside-canonical.
per_sample = []
cell_meta = {}
coverage_rows = []
t0 = time.time()

for ds in DATASETS:
    eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    y_test = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')
    n_test = len(y_test)
    m_conf = np.ones(n_test, dtype=bool)
    m_conf[eval_idx] = False
    y_c, y_conf = y_test[eval_idx], y_test[m_conf]
    print(f'\n{ds}: n_test={n_test}, conformal surface={m_conf.sum()}, canonical={len(eval_idx)}; '
          f'canonical class counts={np.bincount(y_c, minlength=5).tolist()}')

    for model_name in MODELS_PER_DATASET:
        P = np.load(Path(REPO) / 'calibrators' / ds / f'{model_name}_test_proba_calibrated.npy')
        P_conf, P_c = P[m_conf], P[eval_idx]
        y_pred = P_c.argmax(axis=1)
        c1 = P_c[np.arange(len(y_pred)), y_pred].astype(np.float64)
        c2, c2_ret = c2_lookup[(ds, model_name)][:, 0], c2_lookup[(ds, model_name)][:, 1]
        thr, fb, npc = mondrian_conformal_thresholds(P_conf, y_conf, ALPHA_PRIMARY)
        c3 = component_3_safe(P_c, y_pred, thr)
        scts = geo_mean(c1, c2, c3) * 100
        correct = (y_pred == y_c).astype(int)
        tau = np.array([thr[int(p)] for p in y_pred])
        for i in range(len(y_pred)):
            per_sample.append({'protocol': 'v2', 'dataset': ds, 'model': model_name, 'sample_position': i,
                               'true_class': int(y_c[i]), 'pred_class': int(y_pred[i]), 'correct': int(correct[i]),
                               'c1': float(c1[i]), 'c2': float(c2[i]), 'c2_retention': float(c2_ret[i]),
                               'c3': float(c3[i]), 'tau_pred': float(tau[i]), 'scts': float(scts[i])})
        cell_meta[('v2', ds, model_name)] = {
            'thresholds': thr, 'fallback': fb, 'n_per_pred_class': npc,
            'cliff': cliff_fractions(P_conf, y_conf),
            'coverage': empirical_coverage_mondrian(P_c, y_c, thr), 'n_surface': int(m_conf.sum())}
        for a in ALPHAS:
            thr_a, fb_a, _ = mondrian_conformal_thresholds(P_conf, y_conf, a)
            coverage_rows.append({'protocol': 'v2', 'dataset': ds, 'model': model_name, 'alpha': a,
                                  'nominal_coverage': 1 - a,
                                  'empirical_coverage_mondrian': empirical_coverage_mondrian(P_c, y_c, thr_a),
                                  'mean_threshold': float(np.mean(list(thr_a.values()))),
                                  'n_fallback_classes': len(fb_a),
                                  'mean_scts': float((geo_mean(c1, c2, component_3_safe(P_c, y_pred, thr_a)) * 100).mean())})
        print(f'  {model_name:18s} acc={correct.mean():.3f} cov={cell_meta[("v2", ds, model_name)]["coverage"]:.3f} '
              f'mean_scts={scts.mean():6.2f} tau={[round(thr[c], 3) for c in range(5)]}')

print(f'\nv2 protocol: {len(per_sample)} rows; {(time.time() - t0) / 60:.1f} min')


In [ ]:
# Strict protocol (07e/07f): class-aware 80/20 split of the calibration partition; hybrid calibrators refit on
# the 80% strict slice; Mondrian thresholds, cliff fractions and support counts measured on the 20% holdout.
# Nothing in this protocol sees the test distribution before the canonical set is scored.
t0 = time.time()
strict_meta = {}

for ds in DATASETS:
    proc = Path(REPO) / 'data' / 'processed' / ds
    cal_dir = Path(REPO) / 'calibrators' / ds
    y_calib = np.load(proc / 'y_calib_5class.npy')
    strict_idx, holdout_idx, rare = strict_split(y_calib)
    cached_idx = cal_dir / 'X_calib_strict_indices.npy'
    if cached_idx.exists():
        prev = np.load(cached_idx)
        if not np.array_equal(prev, strict_idx):
            print(f'  {ds}: recomputed strict split differs from cached 07e split; using the cached split')
            strict_idx = prev
            holdout_idx = np.setdiff1d(np.arange(len(y_calib)), strict_idx)
    else:
        np.save(cached_idx, strict_idx)
        np.save(cal_dir / 'X_mondrian_holdout_indices.npy', holdout_idx)
    y_h = y_calib[holdout_idx]
    eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    y_test = np.load(proc / 'y_test_5class.npy')
    y_c = y_test[eval_idx]
    print(f'\n{ds}: calib={len(y_calib)} strict={len(strict_idx)} holdout={len(holdout_idx)} '
          f'rare-all-in-strict={[CLASS_NAMES_5[c] for c in rare]} holdout class counts={np.bincount(y_h, minlength=5).tolist()}')

    for model_name in MODELS_PER_DATASET:
        p_test_path = cal_dir / f'{model_name}_test_proba_strict.npy'
        p_hold_path = cal_dir / f'{model_name}_holdout_proba_strict.npy'
        if p_test_path.exists() and p_hold_path.exists():
            P_test, P_h = np.load(p_test_path), np.load(p_hold_path)
            refit = 'cached'
        else:
            p_cal_raw = np.load(find_proba_file(ds, model_name, 'calib'))
            p_test_raw = np.load(find_proba_file(ds, model_name, 'test'))
            y_s = y_calib[strict_idx]
            counts = Counter(y_s.tolist())
            cals, strats = {}, {}
            P_test = np.zeros_like(p_test_raw, dtype=np.float64)
            P_h = np.zeros((len(holdout_idx), p_cal_raw.shape[1]), dtype=np.float64)
            for c in range(p_cal_raw.shape[1]):
                cal, st = fit_calibrator(p_cal_raw[strict_idx, c], (y_s == c).astype(int), counts.get(c, 0))
                cals[c], strats[c] = cal, st
                P_test[:, c] = apply_calibrator(cal, st, p_test_raw[:, c])
                P_h[:, c] = apply_calibrator(cal, st, p_cal_raw[holdout_idx, c])
            for arr in (P_test, P_h):
                rs = arr.sum(axis=1, keepdims=True)
                arr /= np.where(rs == 0, 1, rs)
            np.save(p_test_path, P_test)
            np.save(p_hold_path, P_h)
            joblib.dump({'calibrators': cals, 'strategies': strats, 'calib_counts': dict(counts),
                         'n_classes': p_cal_raw.shape[1], 'platt_threshold': PLATT_THRESHOLD},
                        cal_dir / f'{model_name}_hybrid_strict.joblib')
            refit = 'refit'
        P_c = P_test[eval_idx]
        y_pred = P_c.argmax(axis=1)
        c1 = P_c[np.arange(len(y_pred)), y_pred].astype(np.float64)
        c2, c2_ret = c2_lookup[(ds, model_name)][:, 0], c2_lookup[(ds, model_name)][:, 1]
        thr, fb, npc = mondrian_conformal_thresholds(P_h, y_h, ALPHA_PRIMARY)
        c3 = component_3_safe(P_c, y_pred, thr)
        scts = geo_mean(c1, c2, c3) * 100
        correct = (y_pred == y_c).astype(int)
        tau = np.array([thr[int(p)] for p in y_pred])
        for i in range(len(y_pred)):
            per_sample.append({'protocol': 'strict', 'dataset': ds, 'model': model_name, 'sample_position': i,
                               'true_class': int(y_c[i]), 'pred_class': int(y_pred[i]), 'correct': int(correct[i]),
                               'c1': float(c1[i]), 'c2': float(c2[i]), 'c2_retention': float(c2_ret[i]),
                               'c3': float(c3[i]), 'tau_pred': float(tau[i]), 'scts': float(scts[i])})
        cell_meta[('strict', ds, model_name)] = {
            'thresholds': thr, 'fallback': fb, 'n_per_pred_class': npc,
            'cliff': cliff_fractions(P_h, y_h),
            'coverage': empirical_coverage_mondrian(P_c, y_c, thr), 'n_surface': int(len(holdout_idx))}
        for a in ALPHAS:
            thr_a, fb_a, _ = mondrian_conformal_thresholds(P_h, y_h, a)
            coverage_rows.append({'protocol': 'strict', 'dataset': ds, 'model': model_name, 'alpha': a,
                                  'nominal_coverage': 1 - a,
                                  'empirical_coverage_mondrian': empirical_coverage_mondrian(P_c, y_c, thr_a),
                                  'mean_threshold': float(np.mean(list(thr_a.values()))),
                                  'n_fallback_classes': len(fb_a),
                                  'mean_scts': float((geo_mean(c1, c2, component_3_safe(P_c, y_pred, thr_a)) * 100).mean())})
        print(f'  {model_name:18s} {refit:6s} acc={correct.mean():.3f} cov={cell_meta[("strict", ds, model_name)]["coverage"]:.3f} '
              f'mean_scts={scts.mean():6.2f} tau={[round(thr[c], 4) for c in range(5)]}')

print(f'\nboth protocols: {len(per_sample)} rows (expect {2 * len(DATASETS) * 6 * 1000}); {(time.time() - t0) / 60:.1f} min')


In [ ]:
df = pd.DataFrame(per_sample)
df['scts_c1c2'] = geo_mean(df.c1.values, df.c2.values) * 100
df['scts_c1c3'] = geo_mean(df.c1.values, df.c3.values) * 100
df['architecture'] = df.model.str.split('_').str[0]
df['variant'] = df.model.str.split('_', n=1).str[1]
df.to_csv(TABLES / f'{PREFIX}_per_sample.csv', index=False)
print(f'per-sample table: {df.shape}; saved {PREFIX}_per_sample.csv')

# Consistency with the committed NSL per-sample files (same protocol, same inputs => identical values).
for proto, fname in [('v2', 'scts_v2_canonical.csv'), ('strict', 'scts_v2_canonical_strict_safe.csv')]:
    p = TABLES / fname
    if p.exists():
        o = pd.read_csv(p)[['dataset', 'model', 'sample_position', 'c1', 'c2', 'c3', 'scts', 'correct']]
        m = df[df.protocol == proto].merge(o, on=['dataset', 'model', 'sample_position'], suffixes=('', '_old'))
        print(f'vs {fname}: {len(m)} rows; max|dc1|={np.abs(m.c1 - m.c1_old).max():.1e} '
              f'max|dc2|={np.abs(m.c2 - m.c2_old).max():.1e} max|dc3|={np.abs(m.c3 - m.c3_old).max():.1e} '
              f'max|dscts|={np.abs(m.scts - m.scts_old).max():.1e} correct mismatches={(m.correct != m.correct_old).sum()}')

# c3 is a deterministic function of c1 and the predicted class: c3 = clip(1 - (1 - c1) / tau_pred, 0, 1).
identity_rows = []
for (proto, ds), g in df.groupby(['protocol', 'dataset']):
    nz = g.tau_pred > 1e-9
    recon = np.clip(1 - (1 - g.c1[nz]) / g.tau_pred[nz], 0, 1)
    identity_rows.append({'protocol': proto, 'dataset': ds,
                          'max_abs_reconstruction_error': float(np.abs(recon - g.c3[nz]).max()),
                          'corr_c1_c3': float(np.corrcoef(g.c1, g.c3)[0, 1]),
                          'frac_c3_equals_c1_1e-6': float((np.abs(g.c1 - g.c3) < 1e-6).mean()),
                          'frac_c3_saturated_at_1': float((g.c3 >= 1 - 1e-6).mean()),
                          'frac_c3_floored_at_0': float((g.c3 <= 1e-6).mean()),
                          'corr_c1_c2': float(np.corrcoef(g.c1, g.c2)[0, 1])})
df_identity = pd.DataFrame(identity_rows)
df_identity.to_csv(TABLES / f'{PREFIX}_c3_identity.csv', index=False)
print(df_identity.round(4).to_string(index=False))


In [ ]:
# Per cell: Pearson(score, correctness) and AUROC(score as ranker of correctness) for every score. AUROC is
# reported alongside Pearson because correctness is binary and Pearson is depressed by range restriction in
# high-accuracy cells; AUROC depends only on the ranking.
# plus paired differences against c1 (composite minus calibration alone) on identical resamples.
# CIs are percentile bootstrap over samples (B=N_BOOTSTRAP). Cells with fewer than MIN_MINORITY_FOR_CI
# misclassified or correct samples get point estimates only.
t0 = time.time()
abl_rows = []
for (proto, ds, model_name), g in df.groupby(['protocol', 'dataset', 'model']):
    g = g.sort_values('sample_position')
    y = g.correct.values.astype(np.float64)
    n, n_err = len(y), int((y == 0).sum())
    evaluable = min(n_err, n - n_err) >= MIN_MINORITY_FOR_CI
    W1 = np.ones((1, n))
    W = boot_weights(boot_index(n), n) if evaluable else None
    point = {}
    boots = {}
    for s in SCORES:
        x = g[s].values.astype(np.float64)
        point[s] = (float(pearson_w(W1, x, y)[0]), float(auroc_w(W1, x, y)[0]))
        if evaluable:
            boots[s] = (pearson_w(W, x, y), auroc_w(W, x, y))
    for s in SCORES:
        row = {'protocol': proto, 'dataset': ds, 'model': model_name, 'score': s, 'n': n, 'n_misclassified': n_err,
               'ci_evaluable': evaluable, 'pearson': point[s][0], 'auroc': point[s][1]}
        if evaluable:
            lo, hi, k = pct_ci(boots[s][0]); row.update(pearson_lo=lo, pearson_hi=hi, n_boot_valid_pearson=k)
            lo, hi, k = pct_ci(boots[s][1]); row.update(auroc_lo=lo, auroc_hi=hi, n_boot_valid_auroc=k)
            if s != 'c1':
                d_pr = boots[s][0] - boots['c1'][0]
                d_au = boots[s][1] - boots['c1'][1]
                row.update(delta_pearson_vs_c1=point[s][0] - point['c1'][0],
                           delta_pearson_lo=pct_ci(d_pr)[0], delta_pearson_hi=pct_ci(d_pr)[1],
                           delta_auroc_vs_c1=point[s][1] - point['c1'][1],
                           delta_auroc_lo=pct_ci(d_au)[0], delta_auroc_hi=pct_ci(d_au)[1])
        else:
            for k_ in ['pearson_lo', 'pearson_hi', 'auroc_lo', 'auroc_hi']:
                row[k_] = float('nan')
            if s != 'c1':
                row.update(delta_pearson_vs_c1=point[s][0] - point['c1'][0], delta_pearson_lo=float('nan'),
                           delta_pearson_hi=float('nan'), delta_auroc_vs_c1=point[s][1] - point['c1'][1],
                           delta_auroc_lo=float('nan'), delta_auroc_hi=float('nan'))
        abl_rows.append(row)
    print(f'{proto:6s} {ds:15s} {model_name:18s} n_err={n_err:4d} ' +
          ' '.join(f'{s}:r={point[s][0]:+.3f}/auc={point[s][1]:.3f}' for s in ['c1', 'c2', 'c3', 'scts']))

df_abl = pd.DataFrame(abl_rows)
df_abl.to_csv(TABLES / f'{PREFIX}_per_cell.csv', index=False)
print(f'\n{len(df_abl)} rows; {(time.time() - t0) / 60:.1f} min')

def verdict(lo, hi):
    if not np.isfinite(lo) or not np.isfinite(hi):
        return 'not evaluable'
    if lo > 0: return 'composite better'
    if hi < 0: return 'c1 better'
    return 'no difference'

wide = df_abl.pivot_table(index=['protocol', 'dataset', 'model', 'n_misclassified', 'ci_evaluable'],
                          columns='score', values=['pearson', 'auroc']).reset_index()
wide.columns = ['_'.join([c for c in col if c]) if isinstance(col, tuple) else col for col in wide.columns]
d = df_abl[df_abl.score == 'scts'][['protocol', 'dataset', 'model', 'delta_pearson_vs_c1', 'delta_pearson_lo',
                                    'delta_pearson_hi', 'delta_auroc_vs_c1', 'delta_auroc_lo', 'delta_auroc_hi']]
wide = wide.merge(d, on=['protocol', 'dataset', 'model'])
wide['verdict_pearson'] = [verdict(a, b) for a, b in zip(wide.delta_pearson_lo, wide.delta_pearson_hi)]
wide['verdict_auroc'] = [verdict(a, b) for a, b in zip(wide.delta_auroc_lo, wide.delta_auroc_hi)]
wide.to_csv(TABLES / f'{PREFIX}_per_cell_wide.csv', index=False)

summ = []
for (proto, ds), g in wide.groupby(['protocol', 'dataset']):
    summ.append({'protocol': proto, 'dataset': ds, 'n_cells': len(g),
                 'mean_pearson_c1': g.pearson_c1.mean(), 'mean_pearson_c2': g.pearson_c2.mean(),
                 'mean_pearson_c3': g.pearson_c3.mean(), 'mean_pearson_scts': g.pearson_scts.mean(),
                 'mean_auroc_c1': g.auroc_c1.mean(), 'mean_auroc_c2': g.auroc_c2.mean(),
                 'mean_auroc_c3': g.auroc_c3.mean(), 'mean_auroc_scts': g.auroc_scts.mean(),
                 'mean_auroc_scts_c1c2': g.auroc_scts_c1c2.mean(), 'mean_auroc_scts_c1c3': g.auroc_scts_c1c3.mean(),
                 'cells_scts_auroc_above_c1': int((g.auroc_scts > g.auroc_c1).sum()),
                 'cells_composite_better_auroc_ci': int((g.verdict_auroc == 'composite better').sum()),
                 'cells_c1_better_auroc_ci': int((g.verdict_auroc == 'c1 better').sum()),
                 'cells_no_difference_auroc_ci': int((g.verdict_auroc == 'no difference').sum()),
                 'cells_not_evaluable': int((g.verdict_auroc == 'not evaluable').sum())})
df_summ = pd.DataFrame(summ)
df_summ.to_csv(TABLES / f'{PREFIX}_summary.csv', index=False)
pd.set_option('display.width', 250)
print(df_summ.round(3).to_string(index=False))
print()
print(wide[['protocol', 'dataset', 'model', 'n_misclassified', 'auroc_c1', 'auroc_c2', 'auroc_c3', 'auroc_scts',
            'delta_auroc_vs_c1', 'delta_auroc_lo', 'delta_auroc_hi', 'verdict_auroc']].round(3).to_string(index=False))


In [ ]:
# Cell-level flags (07d rules) for both protocols, joined to samples by predicted class, then evaluated as a
# detector of misclassification. All rates use the cluster bootstrap over canonical sample positions; the naive
# Wilson interval treating the six model rows per sample as independent is reported alongside for comparison.
flag_rows = []
for (proto, ds, model_name), meta in cell_meta.items():
    for c in range(5):
        t, n_c, cf = meta['thresholds'][c], meta['n_per_pred_class'][c], meta['cliff'][c]
        s_t, s_c, s_s = flag_threshold(t), flag_cliff(cf), flag_support(n_c)
        flag_rows.append({'protocol': proto, 'dataset': ds, 'model': model_name, 'predicted_class_idx': c,
                          'predicted_class': CLASS_NAMES_5[c], 'mondrian_threshold': t,
                          'is_fallback': c in meta['fallback'], 'n_calib': n_c, 'cliff_fraction': cf,
                          'signal_threshold': s_t, 'signal_cliff': s_c, 'signal_support': s_s,
                          'calib_health': combine_flags(s_t, s_c, s_s)})
df_flags = pd.DataFrame(flag_rows)
df_flags.to_csv(TABLES / f'{PREFIX}_health_cells.csv', index=False)
print('cell-level flag counts')
print(df_flags.groupby(['protocol', 'dataset', 'calib_health']).size().unstack(fill_value=0).reindex(columns=FLAG_ORDER, fill_value=0))

fl = df_flags.set_index(['protocol', 'dataset', 'model', 'predicted_class_idx'])['calib_health']
df['calib_health'] = [fl[(p, d, m, c)] for p, d, m, c in zip(df.protocol, df.dataset, df.model, df.pred_class)]
df.to_csv(TABLES / f'{PREFIX}_per_sample.csv', index=False)

eval_rows, catch_rows, acc_rows = [], [], []
for (proto, ds), g in df.groupby(['protocol', 'dataset']):
    pos = g.sample_position.values
    mis = (g.correct.values == 0)
    red = (g.calib_health.values == 'red')
    nongreen = (g.calib_health.values != 'green')
    ones = np.ones(len(g), dtype=bool)

    # accuracy by flag, cluster CI
    for f in FLAG_ORDER:
        m = g.calib_health.values == f
        k, nn_ = int((m & ~mis).sum()), int(m.sum())
        lo, hi, _ = cluster_boot_rate(m & ~mis, m, pos) if nn_ > 0 else (float('nan'), float('nan'), 0)
        acc_rows.append({'protocol': proto, 'dataset': ds, 'flag': f, 'n_samples': nn_,
                         'accuracy': k / nn_ if nn_ else float('nan'), 'accuracy_lo': lo, 'accuracy_hi': hi,
                         'share_of_samples': nn_ / len(g)})

    # flag as a detector of misclassification
    for name, pred in [('red', red), ('non_green', nongreen)]:
        tp, fn = int((pred & mis).sum()), int((~pred & mis).sum())
        fp, tn = int((pred & ~mis).sum()), int((~pred & ~mis).sum())
        sens = cluster_boot_rate(pred & mis, mis, pos)
        spec = cluster_boot_rate(~pred & ~mis, ~mis, pos)
        prec = cluster_boot_rate(pred & mis, pred, pos)
        base = cluster_boot_rate(pred, ones, pos)
        eval_rows.append({'protocol': proto, 'dataset': ds, 'detector': name,
                          'n': len(g), 'n_misclassified': int(mis.sum()), 'tp': tp, 'fn': fn, 'fp': fp, 'tn': tn,
                          'sensitivity': tp / (tp + fn), 'sensitivity_lo': sens[0], 'sensitivity_hi': sens[1],
                          'specificity': tn / (tn + fp), 'specificity_lo': spec[0], 'specificity_hi': spec[1],
                          'precision': tp / (tp + fp) if tp + fp else float('nan'), 'precision_lo': prec[0], 'precision_hi': prec[1],
                          'balanced_accuracy': 0.5 * (tp / (tp + fn) + tn / (tn + fp)),
                          'flag_rate_all_samples': pred.mean(), 'flag_rate_lo': base[0], 'flag_rate_hi': base[1],
                          'flag_rate_correct_samples': (pred & ~mis).sum() / (~mis).sum()})

    # per true class: RED rate (the manuscript's R2L catch rate is the R2L row on nsl_kdd_v2), with the
    # cluster CI and, for comparison, the naive Wilson interval on n = samples x models
    for c in range(5):
        m = g.true_class.values == c
        if m.sum() == 0:
            continue
        for name, pred in [('red', red), ('non_green', nongreen)]:
            k, nn_ = int((pred & m).sum()), int(m.sum())
            lo, hi, _ = cluster_boot_rate(pred & m, m, pos)
            w = wilson(k, nn_)
            catch_rows.append({'protocol': proto, 'dataset': ds, 'true_class': CLASS_NAMES_5[c], 'detector': name,
                               'n_rows': nn_, 'n_unique_samples': int(len(np.unique(pos[m]))), 'flagged': k,
                               'rate': k / nn_, 'rate_lo_cluster': lo, 'rate_hi_cluster': hi,
                               'rate_lo_wilson_naive': w[0], 'rate_hi_wilson_naive': w[1],
                               'class_accuracy': float((~mis[m]).mean()),
                               'base_rate_all_samples': float(pred.mean()),
                               'base_rate_correct_samples': float((pred & ~mis).sum() / (~mis).sum())})

df_acc = pd.DataFrame(acc_rows); df_acc.to_csv(TABLES / f'{PREFIX}_health_accuracy_by_flag.csv', index=False)
df_eval = pd.DataFrame(eval_rows); df_eval.to_csv(TABLES / f'{PREFIX}_health_detector_metrics.csv', index=False)
df_catch = pd.DataFrame(catch_rows); df_catch.to_csv(TABLES / f'{PREFIX}_health_rate_by_true_class.csv', index=False)

print('\naccuracy by flag'); print(df_acc.round(3).to_string(index=False))
print('\nflag as detector of misclassification')
print(df_eval[['protocol', 'dataset', 'detector', 'n_misclassified', 'sensitivity', 'sensitivity_lo', 'sensitivity_hi',
               'specificity', 'specificity_lo', 'specificity_hi', 'precision', 'balanced_accuracy',
               'flag_rate_all_samples', 'flag_rate_correct_samples']].round(3).to_string(index=False))
print('\nRED rate by true class (cluster CI vs naive Wilson)')
print(df_catch[df_catch.detector == 'red'][['protocol', 'dataset', 'true_class', 'n_rows', 'n_unique_samples', 'rate',
      'rate_lo_cluster', 'rate_hi_cluster', 'rate_lo_wilson_naive', 'rate_hi_wilson_naive', 'class_accuracy',
      'base_rate_all_samples', 'base_rate_correct_samples']].round(3).to_string(index=False))


In [ ]:
df_cov = pd.DataFrame(coverage_rows)
df_cov['coverage_gap'] = df_cov.empirical_coverage_mondrian - df_cov.nominal_coverage
df_cov.to_csv(TABLES / f'{PREFIX}_conformal_coverage.csv', index=False)
cov_summary = df_cov.groupby(['protocol', 'dataset', 'alpha']).agg(
    nominal=('nominal_coverage', 'first'),
    empirical_mean=('empirical_coverage_mondrian', 'mean'),
    empirical_min=('empirical_coverage_mondrian', 'min'),
    empirical_max=('empirical_coverage_mondrian', 'max'),
    mean_threshold=('mean_threshold', 'mean'),
    mean_scts=('mean_scts', 'mean')).reset_index()
cov_summary.to_csv(TABLES / f'{PREFIX}_conformal_coverage_summary.csv', index=False)
print(cov_summary.round(3).to_string(index=False))


In [ ]:
def fmt(x, d=3):
    return 'n/a' if x is None or (isinstance(x, float) and not np.isfinite(x)) else f'{x:.{d}f}'

def md_table(frame, cols, digits=3):
    head = '| ' + ' | '.join(cols) + ' |\n|' + '|'.join(['---'] * len(cols)) + '|\n'
    body = ''
    for _, r in frame[cols].iterrows():
        body += '| ' + ' | '.join(fmt(v, digits) if isinstance(v, (float, np.floating)) else str(v) for v in r.values) + ' |\n'
    return head + body

summary = {
    'timestamp': datetime.now().isoformat(),
    'notebook': '12_trust_score_ablation.ipynb',
    'datasets': DATASETS, 'models_per_dataset': MODELS_PER_DATASET, 'protocols': PROTOCOLS,
    'n_bootstrap': N_BOOTSTRAP, 'bootstrap_seed': BOOTSTRAP_SEED, 'min_minority_for_ci': MIN_MINORITY_FOR_CI,
    'alpha_primary': ALPHA_PRIMARY, 'top_k': TOP_K, 'attack_source_model': ATTACK_SOURCE_MODEL,
    'c2_definition': 'min over {gaussian, fgsm, pgd} of Jaccard(top-10 clean, top-10 perturbed); retention = |A&B|/10 stored alongside',
    'c3_definition': 'clip(1 - (1 - c1) / tau_pred, 0, 1); a deterministic function of c1 and the predicted class',
    'ablation_summary': df_summ.to_dict(orient='records'),
    'c3_identity': df_identity.to_dict(orient='records'),
    'accuracy_by_flag': df_acc.to_dict(orient='records'),
    'detector_metrics': df_eval.to_dict(orient='records'),
    'coverage_summary': cov_summary.to_dict(orient='records'),
    'outputs': sorted(p.name for p in TABLES.glob(f'{PREFIX}_*')),
}
with open(TABLES / f'{PREFIX}_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))

lines = []
lines.append('# Trust-score component ablation and health-flag evaluation, three datasets, two protocols\n')
lines.append(f'Generated by notebooks/12_trust_score_ablation.ipynb on {datetime.now():%Y-%m-%d}. '
             f'B = {N_BOOTSTRAP} percentile bootstrap, seed {BOOTSTRAP_SEED}. '
             f'Cells with fewer than {MIN_MINORITY_FOR_CI} misclassified or correct canonical samples carry point estimates only.\n')
lines.append('## 1. What c3 is\n')
lines.append('c3 is computed as clip(1 - (1 - c1) / tau, 0, 1), where tau is the Mondrian threshold of the predicted class. '
             'It is therefore a per-class rescaling of c1, not an independent signal. Reconstruction error, '
             'correlation with c1, and the share of samples where c3 equals c1 to 1e-6:\n')
lines.append(md_table(df_identity, ['protocol', 'dataset', 'max_abs_reconstruction_error', 'corr_c1_c3',
                                    'frac_c3_equals_c1_1e-6', 'frac_c3_saturated_at_1', 'frac_c3_floored_at_0'], 4))
lines.append('\n## 2. Component ablation (mean over the six cells per dataset)\n')
lines.append(md_table(df_summ, ['protocol', 'dataset', 'mean_auroc_c1', 'mean_auroc_c2', 'mean_auroc_c3', 'mean_auroc_scts',
                                'mean_auroc_scts_c1c2', 'cells_scts_auroc_above_c1', 'cells_composite_better_auroc_ci',
                                'cells_c1_better_auroc_ci', 'cells_no_difference_auroc_ci', 'cells_not_evaluable']))
lines.append('\nPearson, for continuity with the manuscript:\n')
lines.append(md_table(df_summ, ['protocol', 'dataset', 'mean_pearson_c1', 'mean_pearson_c2', 'mean_pearson_c3', 'mean_pearson_scts']))
lines.append('\n## 3. Per-cell AUROC and paired difference against c1\n')
lines.append(md_table(wide.sort_values(['protocol', 'dataset', 'model']),
                      ['protocol', 'dataset', 'model', 'n_misclassified', 'auroc_c1', 'auroc_c2', 'auroc_c3', 'auroc_scts',
                       'delta_auroc_vs_c1', 'delta_auroc_lo', 'delta_auroc_hi', 'verdict_auroc']))
lines.append('\n## 4. Conformal coverage on the canonical set\n')
lines.append(md_table(cov_summary, ['protocol', 'dataset', 'alpha', 'nominal', 'empirical_mean', 'empirical_min', 'empirical_max',
                                    'mean_threshold', 'mean_scts']))
lines.append('\n## 5. Health flag: accuracy by flag\n')
lines.append(md_table(df_acc, ['protocol', 'dataset', 'flag', 'n_samples', 'share_of_samples', 'accuracy', 'accuracy_lo', 'accuracy_hi']))
lines.append('\n## 6. Health flag as a detector of misclassification\n')
lines.append(md_table(df_eval, ['protocol', 'dataset', 'detector', 'n_misclassified', 'sensitivity', 'sensitivity_lo', 'sensitivity_hi',
                                'specificity', 'specificity_lo', 'specificity_hi', 'precision', 'balanced_accuracy',
                                'flag_rate_all_samples', 'flag_rate_correct_samples']))
lines.append('\n## 7. RED rate by true class, cluster-bootstrap CI over canonical positions versus naive Wilson on rows\n')
lines.append(md_table(df_catch[df_catch.detector == 'red'],
                      ['protocol', 'dataset', 'true_class', 'n_rows', 'n_unique_samples', 'rate', 'rate_lo_cluster', 'rate_hi_cluster',
                       'rate_lo_wilson_naive', 'rate_hi_wilson_naive', 'class_accuracy', 'base_rate_all_samples',
                       'base_rate_correct_samples']))
lines.append('\n## 8. Method notes recorded for the manuscript\n')
lines.append(f'- FGSM and PGD perturbations are generated from {ATTACK_SOURCE_MODEL} and transferred to the other five models in each dataset.\n')
lines.append('- PGD projection is onto the L-infinity ball of radius epsilon (clamp to x plus or minus epsilon).\n')
lines.append(f'- c2 is the minimum over the three perturbations of the Jaccard index between clean and perturbed top-{TOP_K} feature sets; '
             'the retention variant (|A and B| / k) is stored in the per-sample table as c2_retention.\n')
lines.append('- v2 protocol: Mondrian thresholds, cliff fractions and support counts are measured on test-outside-canonical. '
             'Strict protocol: on the 20% holdout of the calibration partition, with calibrators refit on the other 80%.\n')
with open(DOCS / f'{PREFIX}_findings.md', 'w') as f:
    f.write('\n'.join(lines))
print(f'saved docs/{PREFIX}_findings.md and results/tables/{PREFIX}_summary.json')
print('outputs:'); print('\n'.join(summary['outputs']))


In [ ]:
os.chdir(REPO)
!git config user.name "Md Anas Biswas"
!git config user.email "anasbiswas@gmail.com"

import nbformat as _nbf
_nb_path = Path(REPO) / 'notebooks' / '12_trust_score_ablation.ipynb'
if _nb_path.exists():
    _nb = _nbf.read(_nb_path, 4)
    for _c in _nb.cells:
        if _c.cell_type == 'code':
            _c.outputs, _c.execution_count = [], None
    _nbf.write(_nb, _nb_path)
    print(f'outputs stripped: {_nb_path.relative_to(REPO)}')
else:
    raise FileNotFoundError(f'{_nb_path} not found: save this notebook under notebooks/ before committing')

!git add notebooks/12_trust_score_ablation.ipynb
!git add results/tables/trust_ablation_*.csv results/tables/trust_ablation_summary.json
!git add docs/trust_ablation_findings.md
!git add calibrators/*/*_hybrid_strict.joblib
!git status --short | head -40
!git commit -m "Notebook 12: SCTS-v2 component ablation on all three datasets under v2 and strict protocols; c3 identity check; health flag evaluated as a misclassification detector with cluster-bootstrap CIs; conformal coverage by protocol"
!git push origin main
!git log --oneline -3
